# TP 3 - Agent PydanticAI


---
## 0. Configuration partagée


In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

from shared.agent_utils import run_agent_realtime_logging

# À dé-commenter au fur et à mesure que vous implémentez les outils
# (et mettre à jour le __init__.py dans shared)

#from shared.agent_tools import (
#    tool_get_current_date,
#    tool_geocode_location,
#    tool_get_weather,
#    tool_search_nearby,
#    tool_retrieve_docs,
#    web_search,
#    web_extract
#)
from shared.config import ROOT_DIR, google_model_settings, project_settings

**TODO — Configuration partagée et périmètre de code**

Fichiers à modifier :
- `shared/agent_tools.py`
- `TP3_travel_planner_Agent/3_1_tooling_assistant.ipynb`

La fonction principale à coder est `run_agent_realtime_logging` qui exécute un agent avec un prompt donné, en enregistrant la trace complète (étapes, outils appelés, réponses). La partie traceback est déjà fournie dans `shared/agent_utils.py`.

Ce TP s'appuie sur plusieurs outils externes et internes :

`tool_get_current_date` : fonction qui convertit les dates relatives en dates calendrier explicites

`tool_geocode_location` : fonction qui transforme un lieu texte en coordonnées géographiques

`tool_get_weather` : fonction qui retourne une prévision météo sur une plage de dates

`tool_retrieve_docs` : fonction qui récupère les passages RAG internes les plus pertinents

`tool_search_nearby` : fonction qui cherche des lieux proches autour de coordonnées avec filtres

`web_search` : fonction qui lance une recherche web externe

`web_extract` : fonction qui lit le contenu d'une page web trouvée

`Place` : classe qui représente un lieu avec un nom et des coordonnées

Vous pouvez coder ces fonctions en même temps que le développement de l'agent, au fur et à mesure des cas d'usage.


In [ ]:
# TODO : configurer le modèle et les settings partagés pour tous les use cases
model = GoogleModel(
    model_name=...,
    provider=...,
)

Compléter le prompt système ci-dessous, nous y ajouterons en dessous les différentes consignes d'utilisation des outils au fur et à mesure des cas d'usage.

Conseil : Demandez à l'agent de sourcer chaque affirmation factuelle avec le résultat d'outil, par exemple en ajoutant `[tool_name : source]` à la fin de chaque fait. (Exemple : `[web_search : wikipedia.com/Rome]`)

In [ ]:
# TODO : Écrire le prompt système de base

base_system_prompt = """

# RÈGLES

Tu es ...

## Règles générales

### Usage des outils et du contexte
- ...

### Style de réponse
- ...

### Format attendu
1) ...


## Consignes spécifiques par outil

"""


---

## 1. Cas d'usage 1 - Rome en 4 jours

**Objectif** : Prompter et outiller l'agent pour résoudre l'use case que nous avons abordé sur les autres TP.

Exemple d'ordre d'utilisation des outils : date -> géocodage -> météo -> docs -> réponse.
(**Ne pas hard-coder l'ordre ou l'utilisation spécifique d'outils**, l'agent doit pouvoir définir par lui même la stratégie de recherche et d'appel d'outils !)

Vérification concrète dans la trace: chaque recommandation doit être justifiée par au moins un résultat d'outil et respecter durée/budget/préférences.

In [ ]:
prompt_use_case_1 = (
    "Je vais à Rome la semaine prochaine pour 4 jours (du jeudi au dimanche), arrivée le matin, départ le soir. "
    "Fais un plan de 4 jours avec un budget de 300 EUR pour les sorties et restaurants. "
    "Évite les zones trop touristiques et privilégie les lieux confidentiels."
)

In [ ]:
system_prompt_extended = base_system_prompt + """
### tool_...
- ...

### tool_...
- ...

### tool_...
- ...

### tool_...
- ...
"""

tools_use_case_1 = [...]

agent_use_case_1 = Agent(
    model=...,
    instructions=...,
    model_settings=...,
    tools=...,
)

result_use_case_1 = await run_agent_realtime_logging(
    agent=...,
    prompt=...,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_use_case_1.log",
    max_steps=12,
)


In [ ]:
print(result_use_case_1.output)


---

## 2. Cas d'usage 2 - Meilleure période Paris -> New York

Objectif : On montre la polyvalence de l'agent en attaquant d'autres types de questions dans le même domaine de la planification de voyage.

On peut tester plusieurs questions tant que l'agent est capable d'y répondre avec les outils qu'on lui donne !

Dans cet exemple :

Travailler un raisonnement long horizon (plusieurs mois).
`tool_get_weather` ne sert que pour le court terme; la preuve principale doit venir de `web_search` (pour chercher des pages web) et `web_extract` (pour lire des pages web).

Vous pouvez insister sur le fait de faire plusieurs recherches web.

In [ ]:
prompt_use_case_2 = (
    "Trouve la meilleure période dans les 6 prochains mois pour un voyage Paris -> New York. "
    "Compare météo et informations web sur les prix saisonniers, et justifie la recommandation."
    "Précise les prix, le temps de trajet et les conditions météo."
)

In [ ]:
system_prompt_extended = base_system_prompt + """

### tool_...
- ...

### web_search
- ...

### web_extract
- ...

"""

tools_use_case_2 = tools_use_case_1 + [...]

agent_use_case_2 = Agent(
    model=...,
    instructions=...,
    model_settings=...,
    tools=...,
)

result_use_case_2 = await run_agent_realtime_logging(
    agent=...,
    prompt=...,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_use_case_2.log",
    max_steps=12,
)

In [ ]:
print(result_use_case_2.output)

---

## 3. Cas d'usage 3 - Recommandations proches depuis une adresse


In [ ]:
prompt_use_case_3 = (
    "Recommande les meilleurs restaurants et activités près de cette adresse : "
    "10 Rue de la Paix, 75002 Paris, France. " # Vous pouvz tester avec d'autres adresses
    "J'ai un budget de 30 euros pour un repas et 20 euros pour une activité. "
)

In [ ]:
system_prompt_extended = base_system_prompt + """

### tool_...
- ...

### tool_...
- ...

"""

tools_use_case_3 = tools_use_case_2 + [...]

agent_use_case_3 = Agent(
    model=...,
    instructions=...,
    model_settings=...,
    tools=...,
)

result_use_case_3 = await run_agent_realtime_logging(
    agent=...,
    prompt=...,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_use_case_3.log",
    max_steps=12,
)

In [ ]:
print(result_use_case_3.output)